# 🥇 Gold — Top pickup zones

**Purpose.** Rank pickup zones by trip count and total revenue. Useful for understanding where the bulk of yellow-cab demand originates — which is concentrated in a surprisingly small number of Manhattan neighborhoods and the two major airports.

**Source:** `workspace.silver.yellow_taxi`
**Output:** `workspace.gold.top_pickup_zones` (top 50 by trip count)

In [0]:
from pyspark.sql.functions import col, count, sum as spark_sum, avg, round as spark_round

SOURCE_TABLE = "workspace.silver.yellow_taxi"
FULL_TABLE_NAME = "workspace.gold.top_pickup_zones"

silver_df = spark.table(SOURCE_TABLE)

zones_df = (
    silver_df
    .groupBy("pickup_borough", "pickup_zone")
    .agg(
        count("*").alias("trip_count"),
        spark_round(spark_sum("total_amount"), 2).alias("total_revenue"),
        spark_round(avg("fare_amount"), 2).alias("avg_fare"),
        spark_round(avg("trip_distance"), 2).alias("avg_distance_miles"),
        spark_round(avg("trip_duration_minutes"), 2).alias("avg_duration_min"),
    )
    .orderBy(col("trip_count").desc())
    .limit(50)
)

row_count = zones_df.count()
print(f"Top pickup zones: {row_count} rows")

(
    zones_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_TABLE_NAME)
)
print(f"✓ Wrote {row_count} rows to {FULL_TABLE_NAME}")

In [0]:
result = spark.table(FULL_TABLE_NAME)

print("Top 10 pickup zones by trip count:")
display(result.orderBy(col("trip_count").desc()).limit(10))

print("\nTop 10 pickup zones by total revenue:")
display(result.orderBy(col("total_revenue").desc()).limit(10))